# YOLO

- [主程序](#main)

In [1]:
from ultralytics import YOLO
from ultralytics.utils.metrics import DetMetrics, SegmentMetrics
import numpy as np

## 模型训练

In [ ]:
def model_train(data: str, model: str, epochs: int, batch: int, lr: float) -> dict:
    results = YOLO(f"{model}.pt").train(
        data=f"{data}.yaml",
        epochs=epochs,
        batch=batch,
        lr0=lr,
        lrf=lr,
        project=f"runs/{model}-{epochs}-{batch}-{lr}",
        name="train",
        patience=10,
        weight_decay=0.001,
        dropout=0.2
    )
    return results

## 模型验证

In [3]:
def model_val(data: str, project: str, split: str) -> dict:
    results = YOLO(f"runs/{project}/train/weights/best.pt").val(
        data=f"{data}.yaml",
        split=split,
        project=f"runs/{project}",
        name=f"val-{split}"
    )
    return results

## 计算指标

In [4]:
def calculate_detect_metrics(confusion_matrix: np.ndarray, class_names: dict) -> dict:
    """
    从混淆矩阵计算各类的和平均的精确率、召回率、F1 分数以及整体准确率
    
    参数:
    confusion_matrix: numpy 数组, 形状为 (n_classes, n_classes)
    class_names: 类别名称列表
    
    返回:
    dict: 包含所有指标的字典
    """
    n_classes = confusion_matrix.shape[0] - 1
    
    # 初始化结果字典
    results = {
        'per_class': {},
        'mean': {},
        'weighted_avg': {},
        'overall_accuracy': 0
    }
    
    # 每个类别的指标
    precisions = []
    recalls = []
    f1_scores = []
    # 每个类别的样本数
    supports = []
    
    for i in range(n_classes):
        # 真正例
        TP = confusion_matrix[i, i]
        # 假正例
        FP = np.sum(confusion_matrix[:, i]) - TP
        # 假反例
        FN = np.sum(confusion_matrix[i, :]) - TP
        
        # 计算精确率、召回率、F1 Score
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        # 计算当前类别的样本数
        support = np.sum(confusion_matrix[i, :])
        
        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        supports.append(support)
        
        # 存储每个类别的结果
        results['per_class'][class_names[i]] = {
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'support': support
        }

    # 计算平均
    mprecision = np.mean(precisions)
    mrecall = np.mean(recalls)
    mf1_score = np.mean(f1_scores)
    msupport = np.mean(supports)
    results['mean'] = {
        'precision': mprecision,
        'recall': mrecall,
        'f1_score': mf1_score,
        'support': msupport
    }

    # 计算加权平均值
    total_samples = np.sum(supports)
    weighted_precision = np.sum(np.array(precisions) * np.array(supports)) / total_samples
    weighted_recall = np.sum(np.array(recalls) * np.array(supports)) / total_samples
    weighted_f1 = np.sum(np.array(f1_scores) * np.array(supports)) / total_samples
    results['weighted_avg'] = {
        'precision': weighted_precision,
        'recall': weighted_recall,
        'f1_score': weighted_f1,
        'support': total_samples
    }

    # 计算整体准确率
    overall_accuracy = np.trace(confusion_matrix) / np.sum(confusion_matrix)
    results['overall_accuracy'] = overall_accuracy
    
    return results


In [5]:
def calculate_segment_metrics(confusion_matrix: np.ndarray, class_names: dict) -> dict:
    """
    从混淆矩阵计算各类的和平均的像素准确率和交并比
    
    参数:
        confusion_matrix: numpy 数组, 形状为 (n_classes, n_classes)
        class_names: 类别名称列表
    
    返回:
        dict 包含所有指标的字典
    """
    n_classes = confusion_matrix.shape[0] - 1
    
    # 初始化结果字典
    results = {
        'per_class': {},
        'mean': {},
        'weighted_avg': {}
    }
    
    # 每个类别的指标
    pas = []
    ious = []
    # 每个类别的样本数
    supports = []
    
    for i in range(n_classes):
        # 真正例
        TP = confusion_matrix[i, i]
        # 假正例
        FP = np.sum(confusion_matrix[:, i]) - TP
        # 假反例
        FN = np.sum(confusion_matrix[i, :]) - TP
        # 真反例
        
        # 计算像素准确率和交并比
        pa = TP / (TP + FN) if (TP + FN) > 0 else 0
        iou = TP / (TP + FP + FN) if (TP + FN + FN) > 0 else 0
        # 计算当前类别的样本数
        support = np.sum(confusion_matrix[i, :])

        pas.append(pa)
        ious.append(iou)
        supports.append(support)

        # 存储每个类别的结果
        results['per_class'][class_names[i]] = {
            'pa': pa,
            'iou': iou,
            'support': support
        }

    # 计算平均
    mpa = np.mean(pas)
    miou = np.mean(ious)
    msupport = np.mean(supports)
    results['mean'] = {
        'pa': mpa,
        'iou': miou,
        'support': msupport
    }

    # 计算加权平均值
    total_samples = np.sum(supports)
    weighted_pa = np.sum(np.array(pas) * np.array(supports)) / total_samples
    weighted_iou = np.sum(np.array(ious) * np.array(supports)) / total_samples
    results['weighted_avg'] = {
        'pa': weighted_pa,
        'iou': weighted_iou,
        'support': total_samples
    }

    return results

## 打印指标

In [6]:
def print_detect_metrics(detect_metrics: dict) -> None:
    """打印格式化的目标检测精度指标"""
    print(f"{'精度指标':^56}")
    print("=" * 60)
    print(f"{'类别':<14} {'精确率':<7} {'召回率':<7} {'F1分数':<8} {'支持数':<7}")
    print("-" * 60)
    
    for class_name, metrics in detect_metrics['per_class'].items():
        print(f"{class_name:<16} {metrics['precision']:<10.4f} {metrics['recall']:<10.4f} {metrics['f1_score']:<10.4f} {metrics['support']:<10.0f}")
    
    print("-" * 60)
    mean = detect_metrics['mean']
    print(f"{'平均':<14} {mean['precision']:<10.4f} {mean['recall']:<10.4f} {mean['f1_score']:<10.4f} {mean['support']:<10.4f}")
    weighted = detect_metrics['weighted_avg']
    print(f"{'加权平均':<12} {weighted['precision']:<10.4f} {weighted['recall']:<10.4f} {weighted['f1_score']:<10.4f} {weighted['support']:<10.0f}")
    print(f"{'整体准确率':<11} {detect_metrics['overall_accuracy']:<10.4f}")
    print("=" * 60)


In [7]:
def print_segment_metrics(segment_metrics: dict) -> None:
    """打印格式化的语义分割精度指标"""
    print(f"{'精度指标':^56}")
    print("=" * 60)
    print(f"{'类别':<14} {'像素准确率':<5} {'交并比':<7} {'支持数':<7}")
    print("-" * 60)
    
    for class_name, metrics in segment_metrics['per_class'].items():
        print(f"{class_name:<16} {metrics['pa']:<10.4f} {metrics['iou']:<10.4f} {metrics['support']:<10.0f}")
    
    print("-" * 60)
    mean = segment_metrics['mean']
    print(f"{'平均':<14} {mean['pa']:<10.4f} {mean['iou']:<10.4f} {mean['support']:<10.4f}")
    weighted = segment_metrics['weighted_avg']
    print(f"{'加权平均':<12} {weighted['pa']:<10.4f} {weighted['iou']:<10.4f} {weighted['support']:<10.0f}")
    print("=" * 60)

In [8]:
def print_speed_metrics(speed: dict) -> None:
    """打印格式化的速度指标"""
    total_time = speed['preprocess'] + speed['inference'] + speed['loss'] + speed['postprocess']
    print(f"{'速度指标':^56}")
    print("=" * 60)
    print(f"{'总计':<14} {'预处理':<7} {'推理':<8} {'损失':<8} {'后处理':<7}")
    print("-" * 60)
    print(f"{total_time:<16.4f} {speed['preprocess']:<10.4f} {speed['inference']:<10.4f} {speed['loss']:<10.4f} {speed['postprocess']:<10.4f}")
    print("=" * 60)

In [9]:
def print_all_metrics(resluts: DetMetrics | SegmentMetrics, split: str, task: str) -> None:
    matrix = resluts.confusion_matrix.matrix
    names = resluts.confusion_matrix.names
    print("*" * 60)
    print(f"{split:^60}")
    if task == 'detect':
        print_detect_metrics(calculate_detect_metrics(matrix, names))
    elif task == 'segment':
        print_segment_metrics(calculate_segment_metrics(matrix, names))
    print(f"{split:^60}")
    print_speed_metrics(resluts.speed)
    print("*" * 60)

## 集成

In [10]:
def train_val_print(task: str, data: str, model: str, epochs: int, batch: int, lr: float):
    model_train(data, model, epochs, batch, lr)
    project = f"{model}-{epochs}-{batch}-{lr}"
    for split in ["test", "val", "train"]:
        results = model_val(data, project, split)
        print_all_metrics(results, split, task)

## main

### YOLOv11-det

In [15]:
train_val_print("detect", "car", "yolo11n", 150, 4, 0.005)

New https://pypi.org/project/ultralytics/8.3.204 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.200  Python-3.13.7 torch-2.8.0+cu129 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=car.yaml, degrees=10.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.005, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=

### YOLOv12-det

In [ ]:
train_val_print("detect", "car", "yolo12n", 50, 4, 0.01)

New https://pypi.org/project/ultralytics/8.3.204 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.200  Python-3.13.7 torch-2.8.0+cu129 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=car.yaml, degrees=10.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=N

### YOLOv11-seg

In [ ]:
train_val_print("segment", "water", "yolo11n-seg", 50, 4, 0.01)

New https://pypi.org/project/ultralytics/8.3.204 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.200  Python-3.13.7 torch-2.8.0+cu129 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=water.yaml, degrees=10.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, o

### YOLOv12-seg

In [ ]:
train_val_print("segment", "water", "yolov12n-seg", 50, 4, 0.01)

New https://pypi.org/project/ultralytics/8.3.204 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.200  Python-3.13.7 torch-2.8.0+cu129 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=water.yaml, degrees=10.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov12n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, 